In [14]:
# install required packages

%pip install -U langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu transformers torch sentencepiece pypdf

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import all libraries

import os
import re
import torch

from tkinter import Tk, filedialog

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("All libraries imported successfully!")

c:\Users\hp\Data Science and AI\flyrank-ml-internship-wk1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully!


In [ ]:
# set up project folder path

PROJECT_DIR = os.getcwd()

DB_PATH = os.path.join(PROJECT_DIR, "my_pdf_database")

print("Project directory:", PROJECT_DIR)
print("Database directory:", DB_PATH)

Project directory: c:\Users\hp\Data Science and AI\Retrieval-Augmented-Generation-RAG-System-for-Question-Answering
Database directory: c:\Users\hp\Data Science and AI\Retrieval-Augmented-Generation-RAG-System-for-Question-Answering\my_pdf_database


In [ ]:
# embedding model

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 683.17it/s]


Embedding model ready!


In [25]:
# load and creat PDF database

if os.path.exists(os.path.join(DB_PATH, "index.faiss")):

    print("Existing PDF database found!")
    print("Loading saved database...")

    vectorstore = FAISS.load_local(
        DB_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )

    print("Saved database loaded successfully!")

else:

    print("No saved database found.")
    print("Please select your PDF document.")

    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    pdf_path = filedialog.askopenfilename(
        title="Select your PDF",
        filetypes=[("PDF Files", "*.pdf")]
    )

    root.destroy()

    if not pdf_path:
        raise ValueError("No PDF selected!")

    print("Selected:", pdf_path)

    # Load PDF
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    print("Pages loaded:", len(pages))

    # Clean PDF text
    def fix_pdf_text(text):
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    for page in pages:
        page.page_content = fix_pdf_text(page.page_content)

    # Split into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(pages)

    print("Chunks created:", len(chunks))

    # Create FAISS database
    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    # Save database permanently
    vectorstore.save_local(DB_PATH)

    print("PDF database saved successfully!")

Existing PDF database found!
Loading saved database...
Saved database loaded successfully!


In [26]:
# create a retriever for the vectorstore

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever ready!")

Retriever ready!


In [27]:
# load FLAN-T5-small model

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/flan-t5-small"

print("Loading:", model_id)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id
).to(device)

print("Model loaded successfully on:", device)

Loading: google/flan-t5-small


Loading weights: 100%|██████████| 190/190 [00:03<00:00, 53.45it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully on: cpu


In [ ]:
# create RAG question- answering function

def query_rag_v2(question, temperature=0.0, verbose=True):

    # Retrieve relevant documents
    relevant_docs = retriever.invoke(question)

    if not relevant_docs:
        return "I cannot find the answer in the provided report."

    # Extract source pages
    sources = [
        doc.metadata.get("page", "Unknown")
        for doc in relevant_docs
    ]

    # Combine context
    context = "\n".join(
        [doc.page_content for doc in relevant_docs]
    )

    # Create prompt
    input_text = (
        "Answer the question using ONLY the context below.\n"
        "If the answer is missing, reply exactly: "
        "I cannot find the answer in the provided report.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if verbose:
        print("\n--- SOURCE INFORMATION ---")
        print("Retrieved from pages:", sources)

    return answer

In [ ]:
# test the RAG system

print("Testing RAG System...")

question = "How much did the agricultural sector contribute to Kenya's GDP in 2024?"

answer = query_rag_v2(question)

print("\nQUESTION:")
print(question)

print("\nANSWER:")
print(answer)

Testing RAG System...

--- SOURCE INFORMATION ---
Retrieved from pages: [24, 13, 24]

QUESTION:
How much did the agricultural sector contribute to Kenya's GDP in 2024?

ANSWER:
22.5 per cent


### Hallucination Experiment and regulation

In [ ]:
question = "How many tractors were imported from Brazil in 2023?"

results = vectorstore.similarity_search_with_relevance_scores(
    question,
    k=3
)

for i, (doc, score) in enumerate(results):

    print(f"\nResult {i+1}")
    print("Relevance score:", round(score, 4))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print("Retrieved text:", doc.page_content[:300])
    print("-" * 50)


Result 1
Relevance score: 0.284
Page: 79
Retrieved text: National Agriculture Production Report 2025 56 Industrial Crops05
--------------------------------------------------

Result 2
Relevance score: 0.284
Page: 182
Retrieved text: National Agriculture Production Report 2025 159
--------------------------------------------------

Result 3
Relevance score: 0.2838
Page: 1
Retrieved text: National Agriculture Production Report 2025 2
--------------------------------------------------


In [15]:

# IMPROVED RAG FUNCTION WITH HALLUCINATION CONTROL

def query_rag_v2(question, temperature=0.0, verbose=True):

    # Minimum retrieval relevance threshold
    MIN_RELEVANCE = 0.40

    # 1. Retrieve documents with relevance scores
    results = vectorstore.similarity_search_with_relevance_scores(
        question,
        k=3
    )

    if not results:
        return "I cannot find the answer in the provided report."

    # 2. Extract documents and scores
    relevant_docs = [doc for doc, score in results]

    scores = [score for doc, score in results]

    sources = [
        doc.metadata.get("page", "Unknown")
        for doc in relevant_docs
    ]

    # 3. Check retrieval confidence
    best_score = max(scores)

    if verbose:
        print("\n--- RETRIEVAL INFORMATION ---")
        print("Retrieved pages:", sources)
        print("Relevance scores:", scores)
        print("Best relevance:", round(best_score, 4))

    # 4. Reject weak retrieval results
    if best_score < MIN_RELEVANCE:

        if verbose:
            print("Insufficient evidence. Answer rejected.")

        return "I cannot find the answer in the provided report."

    # 5. Combine retrieved context
    context = "\n".join(
        [doc.page_content for doc in relevant_docs]
    )

    # 6. Construct a stricter prompt
    input_text = (
        "You are a factual assistant analyzing an agricultural "
        "statistics report.\n"
        "Answer ONLY using information explicitly stated "
        "in the context.\n"
        "Never guess, calculate missing information, or invent facts.\n"
        "If the context does not explicitly contain the answer, "
        "respond exactly:\n"
        "I cannot find the answer in the provided report.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    # 7. Generate answer
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
question = "How many tractors were imported from Brazil in 2023?"

answer = query_rag_v2(question, verbose=True)

print("\nFINAL ANSWER:")
print(answer)


--- RETRIEVAL INFORMATION ---
Retrieved pages: [79, 182, 1]
Relevance scores: [np.float32(0.2840401), np.float32(0.28401947), np.float32(0.28379107)]
Best relevance: 0.284
Insufficient evidence. Answer rejected.

FINAL ANSWER:
I cannot find the answer in the provided report.


In [19]:
# Test set
test_set = [
    # --- SECTION 1: Food Crops & Staples ---
    {
        "id": 1,
        "question": "What was the total area under maize production and its total output in 2024?",
        "ground_truth": "Retrieved from Table 3.2 (Area, Production and Value of Maize)",
        "target_chunk_id": 11,  # Adjust based on your PDF chunk index
        "is_unanswerable": False,
    },
    {
        "id": 2,
        "question": "How much did the agricultural sector contribute to Kenya's GDP in 2024 according to the report?",
        "ground_truth": "22.5 per cent",
        "target_chunk_id": 3,
        "is_unanswerable": False,
    },
    {
        "id": 3,
        "question": "What were the trends in production volume and value for beans from 2020 to 2024?",
        "ground_truth": "Retrieved from Table 3.10 (Beans Production and Value)",
        "target_chunk_id": 18,
        "is_unanswerable": False,
    },
    {
        "id": 4,
        "question": "What are the core production figures for rice and the performance of the Mwea Irrigation Scheme?",
        "ground_truth": "Retrieved from Tables 3.4 and 3.5 (Rice and Mwea Scheme)",
        "target_chunk_id": 12,
        "is_unanswerable": False,
    },
    {
        "id": 5,
        "question": "What was the total production volume for sorghum and millet in 2024?",
        "ground_truth": "Retrieved from Tables 3.7 and 3.8 (Millet and Sorghum Production)",
        "target_chunk_id": 14,
        "is_unanswerable": False,
    },
    {
        "id": 6,
        "question": "How did green grams and cowpeas production shift across the 2020-2024 period?",
        "ground_truth": "Retrieved from Tables 3.11 and 3.12",
        "target_chunk_id": 21,
        "is_unanswerable": False,
    },

    # --- SECTION 2: Horticulture & Export Crops ---
    {
        "id": 7,
        "question": "What are the leading categories and total volumes of horticulture exports by destination in 2024?",
        "ground_truth": "Retrieved from Tables 4.10 and 4.11 (Horticulture Exports)",
        "target_chunk_id": 53,
        "is_unanswerable": False,
    },
    {
        "id": 8,
        "question": "What is the breakdown of production areas and values for Exotic versus African Leafy Vegetables (ALVs)?",
        "ground_truth": "Retrieved from Tables 4.5 and 4.6",
        "target_chunk_id": 40,
        "is_unanswerable": False,
    },

    # --- SECTION 3: Industrial Crops ---
    {
        "id": 9,
        "question": "What was the total area under coffee and summary of direct sales versus auction volumes up to 2023/24?",
        "ground_truth": "Retrieved from Tables 5.1, 5.1.2, and 5.1.3",
        "target_chunk_id": 57,
        "is_unanswerable": False,
    },
    {
        "id": 10,
        "question": "What were the smallholder and estate production quantities for made tea and green leaf in 2024?",
        "ground_truth": "Retrieved from Table 5.2.1(a)",
        "target_chunk_id": 63,
        "is_unanswerable": False,
    },
    {
        "id": 11,
        "question": "What were the cane deliveries by factory and total sugar production by county in 2024?",
        "ground_truth": "Retrieved from Tables 5.3.1 and 5.3.3",
        "target_chunk_id": 72,
        "is_unanswerable": False,
    },
    {
        "id": 12,
        "question": "What are the tracking figures for sisal production summary and quantities by county?",
        "ground_truth": "Retrieved from Tables 5.4(a) and 5.4(b)",
        "target_chunk_id": 78,
        "is_unanswerable": False,
    },
    {
        "id": 13,
        "question": "What are the seed cotton production yields and lint values recorded between 2020 and 2024?",
        "ground_truth": "Retrieved from Tables 5.5(a) and 5.5(b)",
        "target_chunk_id": 85,
        "is_unanswerable": False,
    },

    # --- SECTION 4: Nuts, Oil Crops, Livestock & Fisheries ---
    {
        "id": 14,
        "question": "What are the area, production, and value numbers for Macadamia and Cashew nuts by county?",
        "ground_truth": "Retrieved from Tables 6.2 and 6.3",
        "target_chunk_id": 104,
        "is_unanswerable": False,
    },
    {
        "id": 15,
        "question": "What was the total milk production and milk intake recorded by processors from 2020 to 2024?",
        "ground_truth": "Retrieved from Table 7.9 (Milk Production and Intake)",
        "target_chunk_id": 131,
        "is_unanswerable": False,
    },
    {
        "id": 16,
        "question": "What are the total quantities and estimated values of meat produced from livestock slaughter in 2024?",
        "ground_truth": "Retrieved from Table 7.11 (Meat Produced and Value)",
        "target_chunk_id": 134,
        "is_unanswerable": False,
    },
    {
        "id": 17,
        "question": "What are the production dynamics and values of fresh fish species across major water bodies up to 2024?",
        "ground_truth": "Retrieved from Tables 8.2 and 8.3",
        "target_chunk_id": 152,
        "is_unanswerable": False,
    },

    # --- SECTION 5: Unanswerable Questions (Hallucination & Honesty Check) ---
    {
        "id": 18,
        "question": "What was the total government expenditure in billion KSh allocated specifically to greenhouse subsidies in Nakuru County for 2024?",
        "ground_truth": "Unanswerable / Not in report",
        "target_chunk_id": None,
        "is_unanswerable": True,
    },
    {
        "id": 19,
        "question": "How many tractors were imported directly from Brazil by large-scale wheat farms in Narok during 2023?",
        "ground_truth": "Unanswerable / Not in report",
        "target_chunk_id": None,
        "is_unanswerable": True,
    },
    {
        "id": 20,
        "question": "What was the precise market share percentage held by private agricultural tech startups in the agritech sector in 2024?",
        "ground_truth": "Unanswerable / Not in report",
        "target_chunk_id": None,
        "is_unanswerable": True,
    },
]

In [20]:

# RETRIEVAL THRESHOLD CALIBRATION

for test in test_set:

    if test["is_unanswerable"]:
        continue

    results = vectorstore.similarity_search_with_relevance_scores(
        test["question"],
        k=3
    )

    scores = [float(score) for doc, score in results]

    print("\nQuestion ID:", test["id"])
    print("Question:", test["question"])
    print("Best relevance score:", round(max(scores), 4))
    print("Retrieved pages:", [
        doc.metadata.get("page", "Unknown")
        for doc, score in results
    ])

    print("-" * 60)



Question ID: 1
Question: What was the total area under maize production and its total output in 2024?
Best relevance score: 0.7117
Retrieved pages: [33, 32, 30]
------------------------------------------------------------

Question ID: 2
Question: How much did the agricultural sector contribute to Kenya's GDP in 2024 according to the report?
Best relevance score: 0.6866
Retrieved pages: [24, 13, 24]
------------------------------------------------------------

Question ID: 3
Question: What were the trends in production volume and value for beans from 2020 to 2024?
Best relevance score: 0.7101
Retrieved pages: [41, 40, 43]
------------------------------------------------------------

Question ID: 4
Question: What are the core production figures for rice and the performance of the Mwea Irrigation Scheme?
Best relevance score: 0.6457
Retrieved pages: [35, 35, 35]
------------------------------------------------------------

Question ID: 5
Question: What was the total production volume fo

In [21]:
# TEST ALL UNANSWERABLE QUESTIONS

for test in test_set:

    if not test["is_unanswerable"]:
        continue

    results = vectorstore.similarity_search_with_relevance_scores(
        test["question"],
        k=3
    )

    scores = [float(score) for doc, score in results]

    print("\nQuestion ID:", test["id"])
    print("Question:", test["question"])
    print("Best relevance score:", round(max(scores), 4))

    answer = query_rag_v2(
        test["question"],
        verbose=False
    )

    print("Generated answer:", answer)
    print("-" * 60)


Question ID: 18
Question: What was the total government expenditure in billion KSh allocated specifically to greenhouse subsidies in Nakuru County for 2024?
Best relevance score: 0.3859
Generated answer: I cannot find the answer in the provided report.
------------------------------------------------------------

Question ID: 19
Question: How many tractors were imported directly from Brazil by large-scale wheat farms in Narok during 2023?
Best relevance score: 0.3925
Generated answer: I cannot find the answer in the provided report.
------------------------------------------------------------

Question ID: 20
Question: What was the precise market share percentage held by private agricultural tech startups in the agritech sector in 2024?
Best relevance score: 0.357
Generated answer: I cannot find the answer in the provided report.
------------------------------------------------------------


In [22]:

# EVALUATE ALL 17 ANSWERABLE QUESTIONS

answerable_results = []

for test in test_set:

    if test["is_unanswerable"]:
        continue

    print("\n" + "=" * 70)
    print("QUESTION ID:", test["id"])
    print("QUESTION:", test["question"])

    answer = query_rag_v2(
        test["question"],
        verbose=True
    )

    print("\nGENERATED ANSWER:")
    print(answer)

    answerable_results.append({
        "ID": test["id"],
        "Question": test["question"],
        "Ground Truth": test["ground_truth"],
        "Generated Answer": answer
    })

# Save results
import pandas as pd

answerable_df = pd.DataFrame(answerable_results)

answerable_df.to_csv(
    "answerable_questions_results.csv",
    index=False
)

print("\nALL 17 ANSWERABLE QUESTIONS COMPLETED!")
print("Results saved successfully.")



QUESTION ID: 1
QUESTION: What was the total area under maize production and its total output in 2024?

--- RETRIEVAL INFORMATION ---
Retrieved pages: [33, 32, 30]
Relevance scores: [np.float32(0.7117235), np.float32(0.68398297), np.float32(0.60273194)]
Best relevance: 0.7117

GENERATED ANSWER:
2,457,025

QUESTION ID: 2
QUESTION: How much did the agricultural sector contribute to Kenya's GDP in 2024 according to the report?

--- RETRIEVAL INFORMATION ---
Retrieved pages: [24, 13, 24]
Relevance scores: [np.float32(0.68664825), np.float32(0.6724936), np.float32(0.6041761)]
Best relevance: 0.6866

GENERATED ANSWER:
22.5 per cent

QUESTION ID: 3
QUESTION: What were the trends in production volume and value for beans from 2020 to 2024?

--- RETRIEVAL INFORMATION ---
Retrieved pages: [41, 40, 43]
Relevance scores: [np.float32(0.71014476), np.float32(0.66606736), np.float32(0.66412026)]
Best relevance: 0.7101

GENERATED ANSWER:
decreased from KSh 88.1 billion in 2023 to KSh 70.4 billion in 20

In [23]:

# CELL 9: RAG EVALUATION WITH PERFORMANCE METRICS

import time
import pandas as pd

# Configuration
k = 3  # Top-k retrieved chunks

results = []

print("Starting RAG evaluation...")
print("=" * 70)

# Expected refusal response
expected_refusal = "I cannot find the answer in the provided report."

# --------------------------------------------------
# 1. EVALUATE EACH TEST QUESTION
# --------------------------------------------------

for test in test_set:

    question = test["question"]
    target_chunk = test["target_chunk_id"]

    # ----------------------------------------------
    # A. Measure Retrieval Latency
    # ----------------------------------------------

    start_retrieval = time.perf_counter()

    relevant_docs = retriever.invoke(question)

    retrieval_time = time.perf_counter() - start_retrieval

    # Extract retrieved pages
    retrieved_pages = [
        doc.metadata.get("page", "Unknown")
        for doc in relevant_docs
    ]

    # Extract retrieved chunk IDs
    retrieved_chunk_ids = [
        doc.metadata.get("chunk_id")
        for doc in relevant_docs
    ]

    # ----------------------------------------------
    # B. Calculate Hit Rate @ K
    # ----------------------------------------------

    top_k_chunks = retrieved_chunk_ids[:k]

    if target_chunk is not None:
        retrieval_hit = int(target_chunk in top_k_chunks)
    else:
        retrieval_hit = None

    # ----------------------------------------------
    # C. Calculate Reciprocal Rank
    # ----------------------------------------------

    mrr = 0.0

    if target_chunk is not None and target_chunk in top_k_chunks:

        rank = top_k_chunks.index(target_chunk) + 1

        mrr = 1.0 / rank

    # ----------------------------------------------
    # D. Measure Generation Latency
    # ----------------------------------------------

    start_generation = time.perf_counter()

    answer = query_rag_v2(
        question,
        temperature=0.0,
        verbose=False
    )

    generation_time = time.perf_counter() - start_generation

    # ----------------------------------------------
    # E. Check Unanswerable Response
    # ----------------------------------------------

    if test["is_unanswerable"]:

        refusal_correct = (
            expected_refusal.lower() in str(answer).lower()
        )

    else:
        refusal_correct = None

    # ----------------------------------------------
    # F. Calculate Total Latency
    # ----------------------------------------------

    total_latency = retrieval_time + generation_time

    # ----------------------------------------------
    # G. Store Evaluation Results
    # ----------------------------------------------

    results.append({

        "ID": test["id"],
        "Question": question,
        "Expected": test["ground_truth"],
        "Generated Answer": answer,

        "Retrieved Pages": retrieved_pages,
        "Retrieved Chunk IDs": retrieved_chunk_ids,
        "Target Chunk": target_chunk,

        "Hit Rate @ K": retrieval_hit,
        "MRR": mrr,

        "Retrieval Latency (s)": retrieval_time,
        "Generation Latency (s)": generation_time,
        "Total Latency (s)": total_latency,

        "Unanswerable": test["is_unanswerable"],
        "Refusal Correct": refusal_correct
    })

    print(
        f"Completed question {test['id']} "
        f"of {len(test_set)} | "
        f"Total latency: {total_latency:.2f}s"
    )


# --------------------------------------------------
# 2. CONVERT RESULTS INTO DATAFRAME
# --------------------------------------------------

results_df = pd.DataFrame(results)


# --------------------------------------------------
# 3. CALCULATE OVERALL EVALUATION METRICS
# --------------------------------------------------

# Exclude questions without a target chunk from retrieval metrics
retrieval_results = results_df.dropna(subset=["Hit Rate @ K"])

if len(retrieval_results) > 0:

    avg_hit_rate = (
        retrieval_results["Hit Rate @ K"].mean() * 100
    )

    avg_mrr = retrieval_results["MRR"].mean()

else:

    avg_hit_rate = 0.0
    avg_mrr = 0.0


# Average latency across all questions
avg_retrieval_latency = results_df[
    "Retrieval Latency (s)"
].mean()

avg_generation_latency = results_df[
    "Generation Latency (s)"
].mean()

avg_total_latency = results_df[
    "Total Latency (s)"
].mean()


# Refusal accuracy for unanswerable questions
unanswerable_results = results_df[
    results_df["Unanswerable"] == True
]

if len(unanswerable_results) > 0:

    refusal_accuracy = (
        unanswerable_results["Refusal Correct"].mean() * 100
    )

else:

    refusal_accuracy = None


# --------------------------------------------------
# 4. DISPLAY FINAL EVALUATION SUMMARY
# --------------------------------------------------

print("\n" + "=" * 70)
print("FINAL RAG EVALUATION SUMMARY")
print("=" * 70)

print(f"Total Test Questions: {len(results_df)}")

print(f"Hit Rate @ {k}: {avg_hit_rate:.2f}%")

print(f"Mean Reciprocal Rank (MRR): {avg_mrr:.4f}")

print(
    f"Average Retrieval Latency: "
    f"{avg_retrieval_latency:.4f} seconds"
)

print(
    f"Average Generation Latency: "
    f"{avg_generation_latency:.4f} seconds"
)

print(
    f"Average Total Latency: "
    f"{avg_total_latency:.4f} seconds"
)

if refusal_accuracy is not None:

    print(
        f"Unanswerable Refusal Accuracy: "
        f"{refusal_accuracy:.2f}%"
    )

print("=" * 70)


# --------------------------------------------------
# 5. SAVE INDIVIDUAL EVALUATION RESULTS
# --------------------------------------------------

results_df.to_csv(
    "rag_evaluation_results.csv",
    index=False
)

print("\nDetailed evaluation results saved to:")
print("rag_evaluation_results.csv")


# --------------------------------------------------
# 6. SAVE OVERALL SUMMARY METRICS
# --------------------------------------------------

summary_df = pd.DataFrame([{

    "Total Questions": len(results_df),
    "Hit Rate @ K (%)": avg_hit_rate,
    "MRR": avg_mrr,
    "Average Retrieval Latency (s)": avg_retrieval_latency,
    "Average Generation Latency (s)": avg_generation_latency,
    "Average Total Latency (s)": avg_total_latency,
    "Unanswerable Refusal Accuracy (%)": refusal_accuracy

}])

summary_df.to_csv(
    "rag_evaluation_summary.csv",
    index=False
)

print("\nSummary metrics saved to:")
print("rag_evaluation_summary.csv")

print("\nEVALUATION COMPLETED SUCCESSFULLY!")


Starting RAG evaluation...
Completed question 1 of 20 | Total latency: 1.30s
Completed question 2 of 20 | Total latency: 2.13s
Completed question 3 of 20 | Total latency: 3.66s
Completed question 4 of 20 | Total latency: 2.87s
Completed question 5 of 20 | Total latency: 2.82s
Completed question 6 of 20 | Total latency: 3.25s
Completed question 7 of 20 | Total latency: 2.82s
Completed question 8 of 20 | Total latency: 3.61s
Completed question 9 of 20 | Total latency: 1.46s
Completed question 10 of 20 | Total latency: 1.23s
Completed question 11 of 20 | Total latency: 1.32s
Completed question 12 of 20 | Total latency: 0.81s
Completed question 13 of 20 | Total latency: 4.24s
Completed question 14 of 20 | Total latency: 1.32s
Completed question 15 of 20 | Total latency: 1.09s
Completed question 16 of 20 | Total latency: 1.93s
Completed question 17 of 20 | Total latency: 1.67s
Completed question 18 of 20 | Total latency: 0.10s
Completed question 19 of 20 | Total latency: 0.09s
Completed que

### Chunk Size Optimization Experiment

In [29]:

# CELL 10: CHUNK SIZE OPTIMIZATION EXPERIMENT

import os
import re
import time
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

chunk_sizes = [300, 700, 1000]
chunk_overlap = 100
k = 3

chunk_experiment_results = []

print("Starting Chunk Size Optimization Experiment")
print("=" * 65)


# --------------------------------------------------
# 2. LOAD PDF PAGES IF NOT ALREADY AVAILABLE
# --------------------------------------------------

if "pages" not in globals() or not pages:

    print("\nPDF pages not found in memory.")
    print("Please select your original PDF.")

    from tkinter import Tk, filedialog

    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    pdf_path = filedialog.askopenfilename(
        title="Select Kenya National Agriculture Production Report 2025",
        filetypes=[("PDF Files", "*.pdf")]
    )

    root.destroy()

    if not pdf_path:
        raise ValueError("No PDF selected!")

    loader = PyPDFLoader(pdf_path)

    pages = loader.load()

    print("PDF loaded successfully!")

    # Clean PDF text
    def fix_pdf_text(text):
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    for page in pages:
        page.page_content = fix_pdf_text(page.page_content)

else:

    print("Using existing PDF pages from memory.")


print("Total PDF pages:", len(pages))


# --------------------------------------------------
# 3. CHECK EMBEDDINGS AND TEST SET
# --------------------------------------------------

if "embeddings" not in globals():
    raise NameError("Embeddings model not initialized. Run your embedding setup cell.")

if "test_set" not in globals() or not test_set:
    raise ValueError("Test set not found. Run your test dataset cell first.")


# --------------------------------------------------
# 4. TEST EACH CHUNK SIZE
# --------------------------------------------------

for chunk_size in chunk_sizes:

    print(f"\nTesting chunk size: {chunk_size}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    chunks = splitter.split_documents(pages)

    print("Chunks created:", len(chunks))

    # Build a separate FAISS index
    vectorstore_test = FAISS.from_documents(
        chunks,
        embeddings
    )

    chunk_retriever = vectorstore_test.as_retriever(
        search_kwargs={"k": k}
    )

    retrieval_times = []
    matching_results = []

    # ----------------------------------------------
    # 5. EVALUATE EACH QUESTION
    # ----------------------------------------------

    for test in test_set:

        question = test["question"]

        start_time = time.perf_counter()

        retrieved_docs = chunk_retriever.invoke(question)

        retrieval_time = time.perf_counter() - start_time

        retrieval_times.append(retrieval_time)

        retrieved_context = " ".join(
            doc.page_content for doc in retrieved_docs
        ).lower()

        expected_information = str(
            test["ground_truth"]
        ).lower().strip()

        expected_keywords = [
            word for word in expected_information.split()
            if len(word) > 3
        ]

        if expected_keywords:

            matching_score = sum(
                keyword in retrieved_context
                for keyword in expected_keywords
            ) / len(expected_keywords)

        else:

            matching_score = 0.0

        matching = matching_score >= 0.5

        matching_results.append(matching)

        chunk_experiment_results.append({

            "Chunk Size": chunk_size,
            "Question ID": test["id"],
            "Question": question,
            "Number of Chunks": len(chunks),
            "Retrieval Latency (s)": retrieval_time,
            "Target Information Match": matching,
            "Keyword Matching Score": matching_score

        })

    # ----------------------------------------------
    # 6. DISPLAY RESULTS
    # ----------------------------------------------

    avg_latency = sum(retrieval_times) / len(retrieval_times)

    matching_accuracy = (
        sum(matching_results) / len(matching_results)
    ) * 100

    print(f"Chunk size: {chunk_size}")
    print(f"Number of chunks: {len(chunks)}")
    print(f"Average retrieval latency: {avg_latency:.4f}s")
    print(f"Target information matching: {matching_accuracy:.2f}%")


# --------------------------------------------------
# 7. SAVE DETAILED RESULTS
# --------------------------------------------------

chunk_results_df = pd.DataFrame(chunk_experiment_results)

chunk_results_df.to_csv(
    "chunk_size_experiment_results.csv",
    index=False
)

print("\nDetailed results saved successfully!")


# --------------------------------------------------
# 8. GENERATE COMPARATIVE SUMMARY
# --------------------------------------------------

chunk_summary = chunk_results_df.groupby(
    "Chunk Size"
).agg(

    Average_Retrieval_Latency=(
        "Retrieval Latency (s)", "mean"
    ),

    Matching_Accuracy=(
        "Target Information Match", "mean"
    ),

    Total_Questions=(
        "Question ID", "count"
    ),

    Total_Chunks=(
        "Number of Chunks", "first"
    )

).reset_index()

chunk_summary["Matching Accuracy (%)"] = (
    chunk_summary["Matching_Accuracy"] * 100
)

print("\n" + "=" * 65)
print("CHUNK SIZE COMPARISON")
print("=" * 65)

print(
    chunk_summary[
        [
            "Chunk Size",
            "Average_Retrieval_Latency",
            "Matching Accuracy (%)",
            "Total_Chunks"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------
# 9. SAVE SUMMARY
# --------------------------------------------------

chunk_summary.to_csv(
    "chunk_size_experiment_summary.csv",
    index=False
)

print("\nCHUNK SIZE OPTIMIZATION COMPLETED!")


Starting Chunk Size Optimization Experiment

PDF pages not found in memory.
Please select your original PDF.
PDF loaded successfully!
Total PDF pages: 202

Testing chunk size: 300
Chunks created: 1436
Chunk size: 300
Number of chunks: 1436
Average retrieval latency: 0.0998s
Target information matching: 35.00%

Testing chunk size: 700
Chunks created: 560
Chunk size: 700
Number of chunks: 560
Average retrieval latency: 0.0267s
Target information matching: 60.00%

Testing chunk size: 1000
Chunks created: 418
Chunk size: 1000
Number of chunks: 418
Average retrieval latency: 0.0261s
Target information matching: 55.00%

Detailed results saved successfully!

CHUNK SIZE COMPARISON
 Chunk Size  Average_Retrieval_Latency  Matching Accuracy (%)  Total_Chunks
        300                   0.099791                   35.0          1436
        700                   0.026737                   60.0           560
       1000                   0.026067                   55.0           418

CHUNK SIZE OP

In [30]:

# CELL 11: HUMAN EVALUATION TEMPLATE

import pandas as pd
import os

print("Generating Human Evaluation Template...")
print("=" * 65)

# --------------------------------------------------
# 1. LOAD EVALUATION RESULTS
# --------------------------------------------------

# Check whether results_df already exists
if "results_df" in globals():

    evaluation_df = results_df.copy()

    print("Using existing evaluation results.")

# Otherwise, load previously saved CSV
elif os.path.exists("rag_evaluation_results.csv"):

    evaluation_df = pd.read_csv(
        "rag_evaluation_results.csv"
    )

    print("Loaded evaluation results from CSV.")

else:

    raise FileNotFoundError(
        "Evaluation results not found. "
        "Please run Cell 9 first or generate "
        "rag_evaluation_results.csv."
    )


# --------------------------------------------------
# 2. CREATE HUMAN EVALUATION TEMPLATE
# --------------------------------------------------

human_evaluation_df = pd.DataFrame({

    # Question identification
    "ID": evaluation_df["ID"],

    # Reference and generated answers
    "Question": evaluation_df["Question"],
    "Ground Truth": evaluation_df["Expected"],
    "Generated Answer": evaluation_df["Generated Answer"],

    # Retrieved context
    "Retrieved Pages": evaluation_df["Retrieved Pages"],

    "Retrieved Chunk IDs": evaluation_df["Retrieved Chunk IDs"],

    # Automated evaluation
    "Hit Rate @ K": evaluation_df["Hit Rate @ K"],
    "MRR": evaluation_df["MRR"],

    # Latency
    "Retrieval Latency (s)": evaluation_df["Retrieval Latency (s)"],
    "Generation Latency (s)": evaluation_df["Generation Latency (s)"],
    "Total Latency (s)": evaluation_df["Total Latency (s)"],

    # Unanswerable questions
    "Unanswerable": evaluation_df["Unanswerable"],
    "Automated Refusal Correct": evaluation_df["Refusal Correct"],

    # Human evaluation fields
    "Human Correctness Score": [None] * len(evaluation_df),
    "Faithfulness Score": [None] * len(evaluation_df),
    "Answer Relevance Score": [None] * len(evaluation_df),
    "Context Relevance Score": [None] * len(evaluation_df),
    "Hallucination Detected": [None] * len(evaluation_df),
    "Human Refusal Correct": [None] * len(evaluation_df),
    "Evaluator Comments": [None] * len(evaluation_df)

})


# --------------------------------------------------
# 3. SAVE HUMAN EVALUATION TEMPLATE
# --------------------------------------------------

human_evaluation_df.to_csv(
    "rag_evaluation_template.csv",
    index=False
)

print("\nHuman evaluation template created successfully!")

print("Saved as: rag_evaluation_template.csv")


# --------------------------------------------------
# 4. DISPLAY TEMPLATE PREVIEW
# --------------------------------------------------

print("\nHUMAN EVALUATION TEMPLATE PREVIEW")
print("=" * 65)

display(human_evaluation_df.head())


# --------------------------------------------------
# 5. DISPLAY SUMMARY
# --------------------------------------------------

print("\nTEMPLATE SUMMARY")
print("=" * 65)

print("Total questions:", len(human_evaluation_df))

print("Total columns:", len(human_evaluation_df.columns))

print("\nHuman grading fields:")

for column in [
    "Human Correctness Score",
    "Faithfulness Score",
    "Answer Relevance Score",
    "Context Relevance Score",
    "Hallucination Detected",
    "Human Refusal Correct",
    "Evaluator Comments"
]:

    print("-", column)

print("\nCELL 11 COMPLETED SUCCESSFULLY!")


Generating Human Evaluation Template...
Using existing evaluation results.

Human evaluation template created successfully!
Saved as: rag_evaluation_template.csv

HUMAN EVALUATION TEMPLATE PREVIEW


,ID,Question,Ground Truth,Generated Answer,Retrieved Pages,Retrieved Chunk IDs,Hit Rate @ K,MRR,Retrieval Latency (s),Generation Latency (s),Total Latency (s),Unanswerable,Automated Refusal Correct,Human Correctness Score,Faithfulness Score,Answer Relevance Score,Context Relevance Score,Hallucination Detected,Human Refusal Correct,Evaluator Comments
0,1,What was the total area under maize production...,"Retrieved from Table 3.2 (Area, Production and...","2,457,025","[33, 32, 30]","[None, None, None]",0.0,0.0,0.064762,1.234349,1.299111,False,None,None,None,None,None,None,None,None
1,2,How much did the agricultural sector contribut...,22.5 per cent,22.5 per cent,"[24, 13, 24]","[None, None, None]",0.0,0.0,0.062820,2.068899,2.131719,False,None,None,None,None,None,None,None,None
2,3,What were the trends in production volume and ...,Retrieved from Table 3.10 (Beans Production an...,decreased from KSh 88.1 billion in 2023 to KSh...,"[41, 40, 43]","[None, None, None]",0.0,0.0,0.088815,3.571177,3.659992,False,None,None,None,None,None,None,None,None
3,4,What are the core production figures for rice ...,Retrieved from Tables 3.4 and 3.5 (Rice and Mw...,3.3 million hectares,"[35, 35, 35]","[None, None, None]",0.0,0.0,0.065979,2.802403,2.868382,False,None,None,None,None,None,None,None,None
4,5,What was the total production volume for sorgh...,Retrieved from Tables 3.7 and 3.8 (Millet and ...,"138,901","[38, 189, 30]","[None, None, None]",0.0,0.0,0.060922,2.762456,2.823378,False,None,None,None,None,None,None,None,None



TEMPLATE SUMMARY
Total questions: 20
Total columns: 20

Human grading fields:
- Human Correctness Score
- Faithfulness Score
- Answer Relevance Score
- Context Relevance Score
- Hallucination Detected
- Human Refusal Correct
- Evaluator Comments

CELL 11 COMPLETED SUCCESSFULLY!


In [31]:

# CELL 12: EXPORT HUMAN EVALUATION TEMPLATE TO CSV AND JSON

import pandas as pd

# 1. Save as CSV
human_evaluation_df.to_csv(
    "rag_human_evaluation_template.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Save as JSON
human_evaluation_df.to_json(
    "rag_human_evaluation_template.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("HUMAN EVALUATION EXPORT COMPLETED!")
print("=" * 55)

print("CSV file: rag_human_evaluation_template.csv")
print("JSON file: rag_human_evaluation_template.json")

print("\nTotal evaluation records:", len(human_evaluation_df))

# 3. Display preview
display(human_evaluation_df.head())

# 4. Download files automatically in Google Colab
try:
    from google.colab import files

    files.download("rag_human_evaluation_template.csv")
    files.download("rag_human_evaluation_template.json")

except ImportError:
    print("\nFiles saved in your current working directory.")
    print("Download them from your Jupyter Notebook file browser.")


HUMAN EVALUATION EXPORT COMPLETED!
CSV file: rag_human_evaluation_template.csv
JSON file: rag_human_evaluation_template.json

Total evaluation records: 20


,ID,Question,Ground Truth,Generated Answer,Retrieved Pages,Retrieved Chunk IDs,Hit Rate @ K,MRR,Retrieval Latency (s),Generation Latency (s),Total Latency (s),Unanswerable,Automated Refusal Correct,Human Correctness Score,Faithfulness Score,Answer Relevance Score,Context Relevance Score,Hallucination Detected,Human Refusal Correct,Evaluator Comments
0,1,What was the total area under maize production...,"Retrieved from Table 3.2 (Area, Production and...","2,457,025","[33, 32, 30]","[None, None, None]",0.0,0.0,0.064762,1.234349,1.299111,False,None,None,None,None,None,None,None,None
1,2,How much did the agricultural sector contribut...,22.5 per cent,22.5 per cent,"[24, 13, 24]","[None, None, None]",0.0,0.0,0.062820,2.068899,2.131719,False,None,None,None,None,None,None,None,None
2,3,What were the trends in production volume and ...,Retrieved from Table 3.10 (Beans Production an...,decreased from KSh 88.1 billion in 2023 to KSh...,"[41, 40, 43]","[None, None, None]",0.0,0.0,0.088815,3.571177,3.659992,False,None,None,None,None,None,None,None,None
3,4,What are the core production figures for rice ...,Retrieved from Tables 3.4 and 3.5 (Rice and Mw...,3.3 million hectares,"[35, 35, 35]","[None, None, None]",0.0,0.0,0.065979,2.802403,2.868382,False,None,None,None,None,None,None,None,None
4,5,What was the total production volume for sorgh...,Retrieved from Tables 3.7 and 3.8 (Millet and ...,"138,901","[38, 189, 30]","[None, None, None]",0.0,0.0,0.060922,2.762456,2.823378,False,None,None,None,None,None,None,None,None



Files saved in your current working directory.
Download them from your Jupyter Notebook file browser.


In [32]:

# CELL 13: INSPECT FAISS CHUNK METADATA

print("FAISS METADATA INSPECTION")
print("=" * 65)

# Retrieve documents from the existing vectorstore
stored_docs = vectorstore.docstore._dict

print("Total indexed chunks:", len(stored_docs))

print("\nFirst 10 indexed chunks:")
print("-" * 65)

for i, (doc_id, doc) in enumerate(stored_docs.items()):

    if i >= 10:
        break

    print(f"\nChunk {i + 1}")
    print("Document ID:", doc_id)
    print("Metadata:", doc.metadata)
    print("Content preview:", doc.page_content[:200])


# Inspect metadata fields
print("\nAVAILABLE METADATA FIELDS")
print("-" * 65)

all_metadata_keys = set()

for doc in stored_docs.values():
    all_metadata_keys.update(doc.metadata.keys())

print(all_metadata_keys)


# Inspect page numbers
page_numbers = sorted(set(
    doc.metadata.get("page")
    for doc in stored_docs.values()
    if doc.metadata.get("page") is not None
))

print("\nUnique PDF page numbers:")
print(page_numbers)

print("\nNumber of unique pages:", len(page_numbers))


FAISS METADATA INSPECTION
Total indexed chunks: 560

First 10 indexed chunks:
-----------------------------------------------------------------

Chunk 1
Document ID: 5cc6cfea-1402-4c63-96ea-56331859976a
Metadata: {'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.2 (Macintosh)', 'creationdate': '2025-10-03T08:22:43+03:00', 'moddate': '2025-10-03T11:47:21+03:00', 'trapped': '/False', 'source': 'C:/Users/hp/Downloads/National-Agriculture-Production-Report-2025.pdf', 'total_pages': 202, 'page': 0, 'page_label': '1'}
Content preview: National Agriculture Production Report 2025 1 National Agriculture Production Report 2025

Chunk 2
Document ID: dba30670-3462-419d-882c-b28c37057722
Metadata: {'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.2 (Macintosh)', 'creationdate': '2025-10-03T08:22:43+03:00', 'moddate': '2025-10-03T11:47:21+03:00', 'trapped': '/False', 'source': 'C:/Users/hp/Downloads/National-Agriculture-Production-Report-2025.pdf', 'total_pa

In [33]:

# CELL 14: SAVE FAISS METADATA INSPECTION

import pandas as pd

metadata_records = []

for doc_id, doc in vectorstore.docstore._dict.items():

    metadata_records.append({

        "Document ID": doc_id,
        "Page Number": doc.metadata.get("page"),
        "Page Label": doc.metadata.get("page_label"),
        "Source": doc.metadata.get("source"),
        "Total Pages": doc.metadata.get("total_pages"),
        "Content Preview": doc.page_content[:300]

    })

metadata_df = pd.DataFrame(metadata_records)

# Save metadata to CSV
metadata_df.to_csv(
    "faiss_metadata_inspection.csv",
    index=False
)

# Save metadata to JSON
metadata_df.to_json(
    "faiss_metadata_inspection.json",
    orient="records",
    indent=4
)

print("FAISS METADATA SAVED SUCCESSFULLY!")
print("=" * 60)

print("Total indexed chunks:", len(metadata_df))
print("Unique PDF pages:", metadata_df["Page Number"].nunique())

print("\nFiles created:")
print("1. faiss_metadata_inspection.csv")
print("2. faiss_metadata_inspection.json")

display(metadata_df.head(10))


FAISS METADATA SAVED SUCCESSFULLY!
Total indexed chunks: 560
Unique PDF pages: 202

Files created:
1. faiss_metadata_inspection.csv
2. faiss_metadata_inspection.json


,Document ID,Page Number,Page Label,Source,Total Pages,Content Preview
0,5cc6cfea-1402-4c63-96ea-56331859976a,0,1,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,National Agriculture Production Report 2025 1 ...
1,dba30670-3462-419d-882c-b28c37057722,1,2,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,National Agriculture Production Report 2025 2
2,1c80f0ed-81d1-462c-bc75-3057f8659eb3,2,i,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,National Agriculture Production Report 2025 i ...
3,fa7bce2a-b265-4c39-9271-0b32be93058d,3,2,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,National Agriculture Production Report 2025ii ...
4,784849e8-da62-4e80-9428-fcf245817552,3,2,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,4.0) available at https:/creativecommons.org/l...
5,47739f23-ae2f-4ff9-bc74-e76c8a4a2968,3,2,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,owned component in the work rests solely with ...
6,9a963c66-ef8e-42d5-b34b-ba9ac47edb4f,4,3,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,National Agriculture Production Report 2025 ii...
7,e3b23ae6-4013-4499-bac3-388d8dbe985d,4,3,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,3.2.1 Beans 17 3.2.2 Cowpeas 21 3.2.3 Green Gr...
8,cb2a0e40-1d6a-41d6-8cff-794babf42ff2,5,4,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,National Agriculture Production Report 2025iv ...
9,de706977-8240-40ff-9ea0-e5ca8a6bb6a5,5,4,C:/Users/hp/Downloads/National-Agriculture-Pro...,202,Area Harvested 72 5.3.3 Cane Deliveries 72 5.3...


In [34]:

# CELL 15: COMPARE TARGET IDs WITH FAISS PAGE NUMBERS

print("TARGET ID ALIGNMENT INSPECTION")
print("=" * 65)

available_pages = set(
    doc.metadata.get("page")
    for doc in vectorstore.docstore._dict.values()
)

for item in test_set:

    target = item["target_chunk_id"]

    print(f"\nQuestion ID: {item['id']}")
    print("Target ID:", target)

    if target is not None:

        print(
            "Target exists as PDF page:",
            target in available_pages
        )

    print("Question:", item["question"])


TARGET ID ALIGNMENT INSPECTION

Question ID: 1
Target ID: 11
Target exists as PDF page: True
Question: What was the total area under maize production and its total output in 2024?

Question ID: 2
Target ID: 3
Target exists as PDF page: True
Question: How much did the agricultural sector contribute to Kenya's GDP in 2024 according to the report?

Question ID: 3
Target ID: 18
Target exists as PDF page: True
Question: What were the trends in production volume and value for beans from 2020 to 2024?

Question ID: 4
Target ID: 12
Target exists as PDF page: True
Question: What are the core production figures for rice and the performance of the Mwea Irrigation Scheme?

Question ID: 5
Target ID: 14
Target exists as PDF page: True
Question: What was the total production volume for sorghum and millet in 2024?

Question ID: 6
Target ID: 21
Target exists as PDF page: True
Question: How did green grams and cowpeas production shift across the 2020-2024 period?

Question ID: 7
Target ID: 53
Target exi

In [35]:

# CELL 16: SAVE TARGET ID ALIGNMENT RESULTS

import pandas as pd

alignment_results = []

available_pages = set(
    doc.metadata.get("page")
    for doc in vectorstore.docstore._dict.values()
)

for item in test_set:

    target = item["target_chunk_id"]

    alignment_results.append({

        "Question ID": item["id"],
        "Question": item["question"],
        "Target Chunk ID": target,
        "Target Exists as PDF Page": (
            target in available_pages
            if target is not None else None
        ),
        "Unanswerable": item["is_unanswerable"]

    })

alignment_df = pd.DataFrame(alignment_results)

# Save CSV
alignment_df.to_csv(
    "target_id_alignment_results.csv",
    index=False
)

# Save JSON
alignment_df.to_json(
    "target_id_alignment_results.json",
    orient="records",
    indent=4
)

print("ALIGNMENT RESULTS SAVED!")
print("CSV: target_id_alignment_results.csv")
print("JSON: target_id_alignment_results.json")

display(alignment_df)


ALIGNMENT RESULTS SAVED!
CSV: target_id_alignment_results.csv
JSON: target_id_alignment_results.json


,Question ID,Question,Target Chunk ID,Target Exists as PDF Page,Unanswerable
0,1,What was the total area under maize production...,11.0,True,False
1,2,How much did the agricultural sector contribut...,3.0,True,False
2,3,What were the trends in production volume and ...,18.0,True,False
3,4,What are the core production figures for rice ...,12.0,True,False
4,5,What was the total production volume for sorgh...,14.0,True,False
5,6,How did green grams and cowpeas production shi...,21.0,True,False
6,7,What are the leading categories and total volu...,53.0,True,False
7,8,What is the breakdown of production areas and ...,40.0,True,False
8,9,What was the total area under coffee and summa...,57.0,True,False
9,10,What were the smallholder and estate productio...,63.0,True,False


In [36]:

# CELL 17: FINAL EVALUATION DATA QUALITY AUDIT

import pandas as pd

print("RAG EVALUATION DATA QUALITY AUDIT")
print("=" * 70)

# 1. Test set analysis
total_questions = len(test_set)

answerable = [
    x for x in test_set
    if not x["is_unanswerable"]
]

unanswerable = [
    x for x in test_set
    if x["is_unanswerable"]
]

valid_targets = [
    x for x in answerable
    if x["target_chunk_id"] is not None
]

missing_targets = [
    x for x in answerable
    if x["target_chunk_id"] is None
]

print("\n1. TEST SET ANALYSIS")
print("Total questions:", total_questions)
print("Answerable questions:", len(answerable))
print("Unanswerable questions:", len(unanswerable))
print("Answerable questions with target IDs:", len(valid_targets))
print("Answerable questions missing target IDs:", len(missing_targets))


# 2. Inspect ground truth
print("\n2. GROUND-TRUTH INSPECTION")
print("-" * 70)

for item in test_set:

    print("\nQuestion ID:", item["id"])
    print("Question:", item["question"])
    print("Ground Truth:", item["ground_truth"])
    print("Target ID:", item["target_chunk_id"])


# 3. Inspect evaluation metrics
if "results_df" in globals():

    print("\n3. RETRIEVAL METRICS")
    print("-" * 70)

    print(
        results_df[
            ["ID", "Target Chunk", "Hit Rate @ K", "MRR"]
        ].to_string(index=False)
    )

    print("\nRefusal evaluation:")

    if "Unanswerable" in results_df.columns:

        refusal_rows = results_df[
            results_df["Unanswerable"] == True
        ]

        print(
            "Unanswerable questions evaluated:",
            len(refusal_rows)
        )

        print(
            "Correct refusals:",
            refusal_rows["Refusal Correct"].sum()
        )


print("\nDATA QUALITY AUDIT COMPLETED!")


RAG EVALUATION DATA QUALITY AUDIT

1. TEST SET ANALYSIS
Total questions: 20
Answerable questions: 17
Unanswerable questions: 3
Answerable questions with target IDs: 17
Answerable questions missing target IDs: 0

2. GROUND-TRUTH INSPECTION
----------------------------------------------------------------------

Question ID: 1
Question: What was the total area under maize production and its total output in 2024?
Ground Truth: Retrieved from Table 3.2 (Area, Production and Value of Maize)
Target ID: 11

Question ID: 2
Question: How much did the agricultural sector contribute to Kenya's GDP in 2024 according to the report?
Ground Truth: 22.5 per cent
Target ID: 3

Question ID: 3
Question: What were the trends in production volume and value for beans from 2020 to 2024?
Ground Truth: Retrieved from Table 3.10 (Beans Production and Value)
Target ID: 18

Question ID: 4
Question: What are the core production figures for rice and the performance of the Mwea Irrigation Scheme?
Ground Truth: Retrie